In [ ]:
%env AWS_PROFILE=platform-developer

In [ ]:
import subprocess
import sys
from pathlib import Path
from typing import Any
import lxml.etree as ET
from utils.marc import parse_single_marc_record

STYLESHEET_PATH = "/Users/brychtas/Documents/GitHub/axiell-collections-xslt/data/stylesheets/axc_to_marcxml_collect.xsl"


def prepare_for_xslt(record_tree: Any) -> ET._Element:
    """Wrap a raw Axiell record tree in the adlibXML/recordList structure the
    XSLT expects, stripping the OAI-PMH default namespace from all elements."""
    root = record_tree.getroot()
    # Strip the OAI default namespace so XSLT template names match
    for elem in root.iter():
        elem.tag = ET.QName(elem).localname
    adlib = ET.Element("adlibXML")
    record_list = ET.SubElement(adlib, "recordList")
    record_list.append(root)
    return adlib


def apply_xslt(
    xml_doc: Any,
    **xslt_params: Any,
):
    parser = ET.XMLParser(remove_blank_text=True)
    xslt_doc = ET.parse(STYLESHEET_PATH, parser)
    transform = ET.XSLT(xslt_doc)

    formatted_params = {
        key: ET.XSLT.strparam(str(value)) for key, value in xslt_params.items()
    }

    return transform(xml_doc, **formatted_params)


In [ ]:
import os
from pathlib import Path
from lxml import etree
import json

def load_all_calm_works():
    calm_works_folder = "/Users/brychtas/Documents/all-raw-calm-works"
    calm_file_names = [f.name for f in Path(calm_works_folder).iterdir() if f.is_file()]
    
    calm_works = {}
    for file_name in calm_file_names:
        with open(os.path.join(calm_works_folder, file_name)) as f:
            parsed_work = json.load(f)
            calm_works[parsed_work["id"]] = parsed_work

    return calm_works


def load_all_axiell_works():
    axiell_works_folder = "/Users/brychtas/Documents/all-raw-axiell-works"
    axiell_file_names = [f.name for f in Path(axiell_works_folder).iterdir() if f.is_file()]
    axiell_works = {}

    for file_name in axiell_file_names:
        if not file_name.endswith(".xml"):
            continue

        parsed_work = etree.parse(os.path.join(axiell_works_folder, file_name))
        identifier = file_name.split(".")[0].replace("_", ":")
        axiell_works[identifier] = parsed_work

    return axiell_works

In [ ]:
calm_works = load_all_calm_works()
print(len(calm_works))

axiell_works = load_all_axiell_works()
print(len(axiell_works))

In [ ]:
from collections import Counter, defaultdict

def find_xml_value(record, field_path: list[str]):
    ns = {"oai": "http://www.openarchives.org/OAI/2.0/"}
    
    path = "/".join([f"oai:{val}" for val in field_path])
    element = record.find(path, namespaces=ns)
    if element is not None:
        return element.text


def count_oai_values(field_path: list[str]):
    values = []
    value_identifier_mapping = defaultdict(list)
    
    for identifier, record in axiell_works.items():
        value = find_xml_value(record, field_path)
        values.append(value)
        value_identifier_mapping[value].append(identifier)

    return Counter(values), value_identifier_mapping


def map_axiell_ids_to_calm_ids():
    mapping = {}
    for axiell_id, record in axiell_works.items():
        calm_id = find_xml_value(record, ["PIDother", "PID_other.non-URI_ID"])
        mapping[axiell_id] = calm_id

    return mapping

In [ ]:
axiell_to_calm = map_axiell_ids_to_calm_ids()

In [ ]:
counter, mapping = count_oai_values(["accession_status", "value"])
print(counter, len(counter))
for i, axiell_id in enumerate(mapping["PRIVATE"]):
    print(axiell_id, axiell_to_calm[axiell_id], calm_works.get(axiell_to_calm[axiell_id], {}).get("data", {}).get("AccessStatus"))
    if i > 100:
        break

In [ ]:
counter, mapping = count_oai_values(["Record_progress", "record_progress", "value[@lang='0']"])
print(counter)
for i, axiell_id in enumerate(mapping["draft"]):
    print(axiell_to_calm[axiell_id], calm_works.get(axiell_to_calm[axiell_id], {}).get("data", {}).get("CatalogueStatus"))
    if i > 100:
        break

In [ ]:
counter, mapping = count_oai_values(["Inscription", "inscription.language"])
print(counter.keys())



In [ ]:
counter, mapping = count_oai_values(["record_type", "value[@lang='0']"])
print(counter)
for i, axiell_id in enumerate(mapping["Work"]):
    print(axiell_id, axiell_to_calm[axiell_id], calm_works.get(axiell_to_calm[axiell_id], {}).get("data", {}).get("Level"))
    if i > 10:
        break

In [ ]:
counter, mapping = count_oai_values(["Object_category", "object_category"])
print(counter)

In [ ]:
# All Mimsy works have consecutive identifiers < 20,000.
# No record with consecutive identifier < 20,000 has a predecessor identifier.

mimsy_count = 0
calm_count = 0
missing_pid_count = 0

for identifier, record in axiell_works.items():
    pid_value = find_xml_value(record, ["PIDother", "PID_other.non-URI_ID"])
    mimsy_value = find_xml_value(record, ["Description", "description.note"])


    
    numeric_id = int(identifier.split(":")[1])
    
    if mimsy_value:
        mimsy_count += 1
    if pid_value:
        calm_count += 1

    if numeric_id > 20000 and not (value := find_xml_value(record, ["current_location.context"])) and find_xml_value(record, ["current_location.name"]):
        print("!!!!!!!!!!")
        print(value)
        break
        #print(etree.tostring(record, pretty_print=True, encoding="unicode"))
        
            
    if not pid_value and numeric_id > 20000:
        #print(etree.tostring(record, pretty_print=True, encoding="unicode"))
        missing_pid_count += 1

print(mimsy_count)
print(calm_count)
print(missing_pid_count)

In [ ]:
from adapters.transformers.builders.axiell_work_builder import AxiellWorkBuilder
from datetime import datetime
from io import BytesIO
from pymarc import parse_xml_to_array


def xslt_result_to_record(xslt_result):
    xml_bytes = bytes(xslt_result)
    return parse_xml_to_array(BytesIO(xml_bytes))[0]


i = 0
for work_id, work_doc in axiell_works.items():
    # Skip Mimsy works
    if int(work_id.split(":")[1]) < 20000:
        continue

    i += 1
    if i > 100:
        break
    xslt_result = apply_xslt(prepare_for_xslt(work_doc), control_003="UkLW")
    record = xslt_result_to_record(xslt_result)

    try:
        work = AxiellWorkBuilder(record=record, last_modified=datetime.now()).transform_work()

        if predecessor_identifier in work.state:
            calm_id = work.state.predecessor_identifier.value
            
    except Exception as e:
        print(e)
    # print(work.model_dump(exclude_none=True))


In [ ]:
from utils.elasticsearch import get_client
from core.source import ElasticSource
import json
import os

SOURCE_WORKS_ROOT = "/Users/brychtas/Documents/all-source-works"


es = get_client("read_only", pipeline_date="2025-10-02", es_mode="public")
source = ElasticSource(es, "works-source-2025-10-02", query={"match_all": {}})

for work in source.stream_raw():
    work_id = work['state']['sourceIdentifier']['value'].replace("/", "+")
    work_id_type = work['state']['sourceIdentifier']['identifierType']['id']

    os.makedirs(f"{SOURCE_WORKS_ROOT}/{work_id_type}", exist_ok=True)
    try:
        with open(f"{SOURCE_WORKS_ROOT}/{work_id_type}/{work_id}.json", "w") as f:
            f.write(json.dumps(work))
    except Exception as e:
        print(e)